# UC-06 - Notebook d'exploration (store_sales et store_returns)

Exploration PySpark des deux tables de faits retenues, lues dans la zone Raw du Data Lake HDFS (jeu initial + rafraichissement). Les memes controles sont rejoues par le script `03_exploration.py`.

## 1. Environnement et session Spark

In [1]:
import sys, platform
sys.path.insert(0, '/scripts')
from commun import *
from nettoyage import charger_type, echecs_conversion
spark = session('UC06 - notebook exploration')
print('Python :', platform.python_version(), '| Spark :', spark.version, '| master :', spark.sparkContext.master)
print('Heure  :', heure())

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/11/18 14:49:33 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Python : 3.10.12 | Spark : 4.2.1 | master : local[*]
Heure  : 18/11/2026 14:49:37


## 2. store_sales : lecture, schema, echantillon, volume

In [2]:
ss = charger_type(spark, 'store_sales')
ss.printSchema()
ss.show(5, truncate=30)
print('Lignes :', ss.count())

root
 |-- ss_sold_date_sk: long (nullable = true)
 |-- ss_sold_time_sk: long (nullable = true)
 |-- ss_item_sk: long (nullable = true)
 |-- ss_customer_sk: long (nullable = true)
 |-- ss_cdemo_sk: long (nullable = true)
 |-- ss_hdemo_sk: long (nullable = true)
 |-- ss_addr_sk: long (nullable = true)
 |-- ss_store_sk: long (nullable = true)
 |-- ss_promo_sk: long (nullable = true)
 |-- ss_ticket_number: long (nullable = true)
 |-- ss_quantity: integer (nullable = true)
 |-- ss_wholesale_cost: decimal(7,2) (nullable = true)
 |-- ss_list_price: decimal(7,2) (nullable = true)
 |-- ss_sales_price: decimal(7,2) (nullable = true)
 |-- ss_ext_discount_amt: decimal(7,2) (nullable = true)
 |-- ss_ext_sales_price: decimal(7,2) (nullable = true)
 |-- ss_ext_wholesale_cost: decimal(7,2) (nullable = true)
 |-- ss_ext_list_price: decimal(7,2) (nullable = true)
 |-- ss_ext_tax: decimal(7,2) (nullable = true)
 |-- ss_coupon_amt: decimal(7,2) (nullable = true)
 |-- ss_net_paid: decimal(7,2) (nullable = 

+---------------+---------------+----------+--------------+-----------+-----------+----------+-----------+-----------+----------------+-----------+-----------------+-------------+--------------+-------------------+------------------+---------------------+-----------------+----------+-------------+-----------+-------------------+-------------+----------+
|ss_sold_date_sk|ss_sold_time_sk|ss_item_sk|ss_customer_sk|ss_cdemo_sk|ss_hdemo_sk|ss_addr_sk|ss_store_sk|ss_promo_sk|ss_ticket_number|ss_quantity|ss_wholesale_cost|ss_list_price|ss_sales_price|ss_ext_discount_amt|ss_ext_sales_price|ss_ext_wholesale_cost|ss_ext_list_price|ss_ext_tax|ss_coupon_amt|ss_net_paid|ss_net_paid_inc_tax|ss_net_profit|jeu_source|
+---------------+---------------+----------+--------------+-----------+-----------+----------+-----------+-----------+----------------+-----------+-----------------+-------------+--------------+-------------------+------------------+---------------------+-----------------+----------+----

Lignes : 674211


## 3. Valeurs manquantes (store_sales)

In [3]:
n = ss.count()
manq = ss.agg(*[F.sum(F.col(c).isNull().cast('int')).alias(c) for c in noms('store_sales')]).first()
for c in noms('store_sales'):
    if manq[c]:
        print(f'{c:20s} {manq[c]:>8d} ({100 * manq[c] / n:.2f} %)')

ss_cdemo_sk               250 (0.04 %)
ss_hdemo_sk               308 (0.05 %)
ss_addr_sk                462 (0.07 %)
ss_promo_sk               159 (0.02 %)


## 4. Doublons et cle candidate (store_sales)

In [4]:
cle = CLES['store_sales']
print('Lignes strictement identiques :', n - ss.dropDuplicates(noms('store_sales')).count())
print('Repetitions de la cle', cle, ':', n - ss.dropDuplicates(cle).count())
ss.groupBy(*cle).count().filter('count > 1').orderBy(F.desc('count')).show(5)

Lignes strictement identiques : 0


Repetitions de la cle ['ss_ticket_number', 'ss_item_sk'] : 4182


+----------------+----------+-----+
|ss_ticket_number|ss_item_sk|count|
+----------------+----------+-----+
|            2378|     10003|    4|
|           37842|     10000|    4|
|           13035|     10001|    3|
|            1146|     16513|    3|
|            5725|     10000|    3|
+----------------+----------+-----+
only showing top 5 rows


## 5. Valeurs atypiques (store_sales)

Une valeur negative ou un profit negatif n'est pas une erreur par principe : on les compte avant de decider.

In [5]:
ss.agg(F.min('ss_quantity'), F.max('ss_quantity'), F.min('ss_sales_price'), F.max('ss_sales_price'),
       F.min('ss_net_profit'), F.max('ss_net_profit')).show()
print('Quantite <= 0 :', ss.filter('ss_quantity <= 0').count())
print('Prix de vente = 0 :', ss.filter('ss_sales_price = 0').count())
print('Profit net < 0 :', ss.filter('ss_net_profit < 0').count())

+----------------+----------------+-------------------+-------------------+------------------+------------------+
|min(ss_quantity)|max(ss_quantity)|min(ss_sales_price)|max(ss_sales_price)|min(ss_net_profit)|max(ss_net_profit)|
+----------------+----------------+-------------------+-------------------+------------------+------------------+
|               1|             100|               0.00|             199.30|          -9866.60|           9134.00|
+----------------+----------------+-------------------+-------------------+------------------+------------------+



Quantite <= 0 : 0


Prix de vente = 0 : 3361


Profit net < 0 : 500889


## 6. store_returns : schema, volume, valeurs manquantes, doublons

In [6]:
sr = charger_type(spark, 'store_returns')
sr.printSchema()
print('Lignes :', sr.count())
m = sr.agg(*[F.sum(F.col(c).isNull().cast('int')).alias(c) for c in noms('store_returns')]).first()
print({c: m[c] for c in noms('store_returns') if m[c]})
print('Repetitions de la cle', CLES['store_returns'], ':', sr.count() - sr.dropDuplicates(CLES['store_returns']).count())

root
 |-- sr_returned_date_sk: long (nullable = true)
 |-- sr_return_time_sk: long (nullable = true)
 |-- sr_item_sk: long (nullable = true)
 |-- sr_customer_sk: long (nullable = true)
 |-- sr_cdemo_sk: long (nullable = true)
 |-- sr_hdemo_sk: long (nullable = true)
 |-- sr_addr_sk: long (nullable = true)
 |-- sr_store_sk: long (nullable = true)
 |-- sr_reason_sk: long (nullable = true)
 |-- sr_ticket_number: long (nullable = true)
 |-- sr_return_quantity: integer (nullable = true)
 |-- sr_return_amt: decimal(7,2) (nullable = true)
 |-- sr_return_tax: decimal(7,2) (nullable = true)
 |-- sr_return_amt_inc_tax: decimal(7,2) (nullable = true)
 |-- sr_fee: decimal(7,2) (nullable = true)
 |-- sr_return_ship_cost: decimal(7,2) (nullable = true)
 |-- sr_refunded_cash: decimal(7,2) (nullable = true)
 |-- sr_reversed_charge: decimal(7,2) (nullable = true)
 |-- sr_store_credit: decimal(7,2) (nullable = true)
 |-- sr_net_loss: decimal(7,2) (nullable = true)
 |-- jeu_source: string (nullable = fal

{'sr_cdemo_sk': 21, 'sr_hdemo_sk': 15, 'sr_addr_sk': 42, 'sr_reason_sk': 11}


Repetitions de la cle ['sr_ticket_number', 'sr_item_sk'] : 160


## 7. Conversions de types

Une valeur non vide qui devient NULL apres conversion est comptee.

In [7]:
print('store_sales   :', {c: v for c, v in echecs_conversion(spark, 'store_sales').items() if v} or 'aucun echec')
print('store_returns :', {c: v for c, v in echecs_conversion(spark, 'store_returns').items() if v} or 'aucun echec')
spark.stop()

store_sales   : aucun echec


store_returns : aucun echec


## 8. Constats

Les constats chiffres sont ceux affiches ci-dessus ; leur interpretation et les regles de nettoyage associees sont dans le document des regles de nettoyage.